<a href="https://colab.research.google.com/github/vera2005/MLCV_course1/blob/main/Lesson3/ML_Course_Lesson_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

0. Импорты

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Воспроизводимость
np.random.seed(42)
tf.random.set_seed(42)

Вспомогательные функции

plot_history — рисует графики loss и accuracy для train и val, чтобы глазами увидеть переобучение.

compute_metrics — считает accuracy, precision, recall и F1 (macro и weighted) на тесте и возвращает словарь для сравнения моделей.

show_classification_report — печатает детальный отчёт по метрикам для каждого класса отдельно.

show_confusion_matrix — рисует матрицу ошибок в виде heatmap, чтобы видеть, какие классы модель путает.

In [ ]:
def plot_history(history, title):
    """Строит графики loss и accuracy для train и val."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'], label='train', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='val', linewidth=2)
    axes[0].set_title(f'{title} — Loss')
    axes[0].set_xlabel('Эпоха')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history.history['accuracy'], label='train', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='val', linewidth=2)
    axes[1].set_title(f'{title} — Accuracy')
    axes[1].set_xlabel('Эпоха')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()


def compute_metrics(model, x_test, y_test_true, name):
    """
    Считает метрики: accuracy, precision, recall, F1 (macro и weighted).
    Возвращает словарь с метриками и массив предсказаний.
    """
    y_pred_prob = model.predict(x_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)

    result = {
        'name':        name,
        'accuracy':    accuracy_score(y_test_true, y_pred),
        'precision':   precision_score(y_test_true, y_pred,
                                       average='macro', zero_division=0),
        'recall':      recall_score(y_test_true, y_pred,
                                    average='macro', zero_division=0),
        'f1_macro':    f1_score(y_test_true, y_pred,
                                average='macro', zero_division=0),
        'f1_weighted': f1_score(y_test_true, y_pred,
                                average='weighted', zero_division=0),
        'y_pred':      y_pred
    }

    print(f"\n=== {name} ===")
    print(f"Accuracy:         {result['accuracy']:.4f}")
    print(f"Precision macro:  {result['precision']:.4f}")
    print(f"Recall macro:     {result['recall']:.4f}")
    print(f"F1 macro:         {result['f1_macro']:.4f}")
    print(f"F1 weighted:      {result['f1_weighted']:.4f}")

    return result


def show_classification_report(y_test_true, y_pred, name):
    """Печатает детальный отчёт по каждому классу."""
    print(f"\n--- Classification report: {name} ---")
    print(classification_report(y_test_true, y_pred, digits=4))


def show_confusion_matrix(y_test_true, y_pred, title, cmap='Blues'):
    """Рисует confusion matrix."""
    cm = confusion_matrix(y_test_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                cbar=False, square=True)
    plt.title(title)
    plt.xlabel('Предсказано')
    plt.ylabel('Истина')
    plt.tight_layout()
    plt.show()

Precision — «из тех, кого я назвал +, сколько реально +». Насколько можно верить, когда модель говорит «да».

Recall — «из всех реальных +, сколько я нашёл». Сколько пропустил.

F1 — «баланс между precision и recall». Одним числом, когда важно и не ошибиться, и не пропустить.

Аналогия для запоминания (рыбалка): Precision — «из всего, что я вытащил из пруда, сколько реально рыба, а не мусор?»

Recall — «из всей рыбы в пруду, сколько я поймал?»

1. Загрузка данных

2. Подготовка данных


In [ ]:
# Нормализация: пиксели были 0..255 → станут 0..1
x_train_full =
x_test =

# One-hot encoding меток
y_train_full_cat = to_categorical(y_train_full, 10)
y_test_cat = to_categorical(y_test, 10)

# ВАЖНО: специально берём маленькую выборку,
# чтобы вызвать переобучение у базовой модели

x_train =
y_train =

x_val =
y_val =

print(f"Train: {x_train.shape}")
print(f"Val:   {x_val.shape}")
print(f"Test:  {x_test.shape}")
print(f"Классов: {y_train_full_cat.shape[1]}")

3. Создание базовой модели (для переобучения) БЕЗ РЕГУЛЯРИЗАЦИЙ

In [ ]:
model_base = models.Sequential([
    layers.,
    layers.   # скрытый слой
    layers.   # скрытый слой
    layers.  # выход: 10 классов
])

model_base.compile(

)

history_base = model_base.fit(

)

# Графики обучения
plot_history(history_base, "Базовая модель")

Прочерка метриками

In [ ]:
result_base = compute_metrics(model_base, x_test, y_test, "Базовая модель")

Отчет по классам

In [ ]:
show_classification_report(y_test, result_base['y_pred'], "Базовая модель")

Матрица ошибок

In [ ]:
# Confusion matrix
show_confusion_matrix(
    y_test, result_base['y_pred'],
    title='Confusion Matrix — Базовая модель (переобучена)',
    cmap='Reds'
)

4. Создание ХОРОШЕЙ модели

In [ ]:
model_reg = models.Sequential([
    layers.
    layers.
    layers.
    layers.
    layers.
    layers.
])

model_reg.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:

# EarlyStopping: остановка, когда val_loss перестал падать
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history_reg = model_reg.fit(
    x_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(x_val, y_val),
    callbacks=[early_stop], # спец объект, который Keras вызывает после каждой эпохи обучения.
    #EarlyStopping следит за метрикой и останавливает обучение, если она перестала улучшаться.
)

# Графики обучения
plot_history(history_reg, "Регуляризованная модель")

Проверка метрик

In [ ]:
result_reg = compute_metrics(model_reg, x_test, y_test, "Регуляризованная модель")

ОТчет по классам

In [ ]:
show_classification_report(y_test, result_reg['y_pred'], "Регуляризованная модель")

Матрица ошибок

In [ ]:
# Confusion matrix
show_confusion_matrix(
    y_test, result_reg['y_pred'],
    title='Confusion Matrix — Регуляризованная модель',
    cmap='Greens'
)

5. Сравнение метрик

In [ ]:
print(f"\n{'Метрика':<20}{'Базовая':<15}{'Регуляриз.':<15}{'Δ':<15}")
print("-" * 65)

metrics_to_compare = ['accuracy', 'precision', 'recall', 'f1_macro', 'f1_weighted']

for m in metrics_to_compare:
    base_val = result_base[m]
    reg_val  = result_reg[m]
    delta    = reg_val - base_val
    sign     = '+' if delta >= 0 else ''
    print(f"{m:<20}{base_val:<15.4f}{reg_val:<15.4f}{sign}{delta:<15.4f}")

6. Визуальное сравнение метрик

In [ ]:
# Визуальное сравнение метрик
labels = ['Accuracy', 'Precision\n(macro)', 'Recall\n(macro)',
          'F1\n(macro)', 'F1\n(weighted)']

base_values = [result_base[m] for m in metrics_to_compare]
reg_values  = [result_reg[m]  for m in metrics_to_compare]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, base_values, width, label='Базовая', color='salmon')
bars2 = ax.bar(x + width/2, reg_values,  width, label='Регуляризованная', color='mediumseagreen')

ax.set_ylabel('Значение метрики')
ax.set_title('Сравнение метрик: базовая vs регуляризованная')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0.85, 1.0)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bars in (bars1, bars2):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()